# Analiza streszczeń literartury młodzieżowej z konca XX wieku

## Import bibliotek

In [ ]:

import os
import json
import nltk
import string
import morfeusz2
import numpy as np
import pandas as pd
import operator as op
import itertools as it
from wordcloud import WordCloud
from matplotlib import pyplot as plt
from matplotlib.colors import to_hex
from nltk.util import ngrams
from nltk.tokenize import word_tokenize
from nltk.corpus import PlaintextCorpusReader
from scipy.cluster.hierarchy import dendrogram
from sklearn.cluster import AgglomerativeClustering
from sklearn.decomposition import LatentDirichletAllocation, NMF
from sklearn.metrics.pairwise import cosine_similarity, euclidean_distances
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer

In [ ]:
nltk.download('punkt_tab')
nltk.pathsec.ENFORCE = False

## zdefiniowanie funkcji pomocniczych

In [ ]:
def print_dict(obj):
    print(json.dumps(obj, indent= 4, ensure_ascii= False))

In [ ]:
def create_dir(path):
    if not os.path.exists(path):
        os.mkdir(path)

## Utorzenie korpusu dokumentów


In [ ]:
corpus_dir = "./Literature"
corpus = PlaintextCorpusReader(corpus_dir, r".*\.txt")
files_names = corpus.fileids()

In [ ]:
documents_dict = {}
for file_name in files_names:
    documents_dict[file_name] = corpus.raw(file_name)
print_dict(documents_dict)

In [ ]:
lengths = {}
for file_name in documents_dict:
    lengths[file_name] = {
        "pre": len(word_tokenize(documents_dict[file_name])),
    }
print_dict(lengths)

## Utworzenie macierzy częstości

In [ ]:
documents_df_pre = pd.DataFrame.from_dict(documents_dict, orient='index', columns=['content'])

In [ ]:
matrix_tf = CountVectorizer().fit_transform(documents_df_pre['content'])

In [ ]:
sparsity_tf = (1 - (matrix_tf.nnz / (matrix_tf.shape[0] * matrix_tf.shape[1]))) * 100

In [ ]:
matrices = {
    "pre": {
        "words": matrix_tf.shape[1],
        "sparsity": sparsity_tf
    }
}

## Wstępne przygotowanie dokumentów 

In [ ]:
stopwords_file = open("./stopwords_pl.txt", "r", encoding="utf-8")
stopwords = stopwords_file.read().splitlines()
stopwords = stopwords[4:]  # Skip the first four lines of the file
stopwords_file.close()
stopwords

In [ ]:
def lemmatize(text):
    morfer = morfeusz2.Morfeusz()
    segments = it.groupby(morfer.analyse(text), op.itemgetter(0,1))
    def disambiguate(group):
        pairs = ((len(descr), lemma) for _, _, (_, lemma, descr, _, _, ) in group)
        perpl, lemma = min(pairs)
        return lemma.split(":")[0] if lemma else None
    lemmas = (disambiguate(group) for _, group in segments)
    return " ".join(filter(str.isalpha, lemmas))

In [ ]:
for file_name in documents_dict:
    documents_dict[file_name] = documents_dict[file_name].lower()
    documents_dict[file_name] = "".join([char for char in documents_dict[file_name] if char not in string.punctuation])
    documents_dict[file_name] = lemmatize(documents_dict[file_name])
    documents_dict[file_name] = " ".join([word for word in word_tokenize(documents_dict[file_name], language='polish') if word not in stopwords])
print_dict(documents_dict)

In [ ]:
for file_name in documents_dict:
    lengths[file_name]["post"] = len(word_tokenize(documents_dict[file_name]))
print_dict(lengths)

In [ ]:
lengths = pd.DataFrame.from_dict(lengths, orient='index')
lengths['change'] = lengths['post'] - lengths['pre']
lengths['change_percentage'] = ((lengths['change'] / lengths['pre']) * 100).round(2)
display(lengths)

# Utworzenie Macierzy Częstości

In [ ]:
documents_df_post = pd.DataFrame.from_dict(documents_dict, orient='index', columns=['content'])

In [ ]:
cv = CountVectorizer()
matrix_tf = cv.fit_transform(documents_df_post['content'])
tv = TfidfVectorizer()
matrix_tfidf = tv.fit_transform(documents_df_post['content'])

In [ ]:
sparsity_tf = (1 - (matrix_tf.nnz / (matrix_tf.shape[0] * matrix_tf.shape[1]))) * 100

In [ ]:
matrices["post"] = {
    "words": matrix_tf.shape[1],
    "sparsity": sparsity_tf
}

In [ ]:
matrices = pd.DataFrame.from_dict(matrices, orient="index")
display(matrices)

## Przygotowanie katalogów wyjściowych

In [ ]:
create_dir("./wordclouds")
create_dir("./wordclouds/pre")
create_dir("./wordclouds/post")
create_dir("./topic_models")
create_dir("./clustering")
create_dir("./ngrams")

## Wordclouds

In [ ]:
wordcloud = WordCloud(
    background_color="white",
    max_words=5000,
    contour_width=3,
    contour_color="steelblue"
)

In [ ]:
for index, row in documents_df_pre.iterrows():
    wordcloud.generate(row["content"])
    plt.imshow(wordcloud, interpolation="bilinear")
    plt.axis("off")
    plt.title(index.replace(".txt", ""))
    plt.savefig(f"./wordclouds/pre/{index.replace(".txt", ".png")}")
    plt.close()

In [ ]:
for index, row in documents_df_post.iterrows():
    wordcloud.generate(row["content"])
    plt.imshow(wordcloud, interpolation="bilinear")
    plt.axis("off")
    plt.title(index.replace(".txt", ""))
    plt.savefig(f"./wordclouds/post/{index.replace(".txt", ".png")}")
    plt.close()

## Analiza tematyk 

In [ ]:
def plot_top_words(model, feature_names, n_top_words, title, subplots, colors):
    fig, axes = plt.subplots(*subplots, figsize=(30, 15), sharex=True)
    axes = axes.flatten()
    for topic_idx, topic in enumerate(model.components_):
        top_features_ind = topic.argsort()[-n_top_words:]
        top_features = feature_names[top_features_ind]
        weights = topic[top_features_ind]

        ax = axes[topic_idx]
        ax.barh(top_features, weights, height=0.7, color=colors[topic_idx])
        ax.set_title(f"Temat {topic_idx + 1}", fontdict={"fontsize": 30})
        ax.tick_params(axis="both", which="major", labelsize=20)
        for i in "top right left".split():
            ax.spines[i].set_visible(False)
        fig.suptitle(title, fontsize=40)

    plt.subplots_adjust(top=0.90, bottom=0.05, wspace=0.90, hspace=0.3)
    plt.savefig("./topic_models/" + str(n_components) + "/topics/" + title.replace(" ", "_") + ".png")
    plt.close()

In [ ]:
def plot_documents(model, matrix, n_components, title, colors):
    documents_topics = pd.DataFrame(model.transform(matrix))
    documents_topics.columns = ["Temat " + str(i) for i in range(1, n_components+1)]
    documents_topics.index = [file_name.replace(".txt", "") for file_name in files_names]
    n_docs = len(documents_topics)
    plt.figure(figsize=(7, 4))
    left = [0] * n_docs
    for i, col in enumerate(documents_topics.columns):
        plt.barh(documents_topics.index, documents_topics[col], left=left, label=col, color=colors[i])
        left = [left[j]+ documents_topics[col].iloc[j] for j in range(n_docs)]
    plt.tight_layout()
    plt.savefig("./topic_models/" + str(n_components) + "/documents/" + title.replace(" ", "_") + ".png")
    plt.close()

In [ ]:
colors = [
    "navy",
    "gold",
    "coral",
    "royalblue",
    "limegreen",
    "firebrick",
    "indigo",
    "olive",
    "salmon",
    "darkcyan",
    "goldenrod",
]
feature_names =  cv.get_feature_names_out()
n_top_features = 20
size = [1,4]
n_components = size[0]*size[1]


In [ ]:
create_dir("./topic_models/"+str(n_components))
create_dir("./topic_models/"+str(n_components)+"/topics")
create_dir("./topic_models/"+str(n_components)+"/documents")

In [ ]:
lda = LatentDirichletAllocation(
    n_components=n_components,
    max_iter=5,
    learning_method="batch",
    learning_offset=50,
    random_state=0
)
lda.fit(matrix_tf)
plot_top_words(lda, feature_names, n_top_features, "Tematu w modeli LDA", size, colors)
plot_documents(lda, matrix_tf, n_components, "Tematu w modeli LDA", colors)

In [ ]:
nmf_fn = NMF(
    n_components=n_components,
    random_state=1, 
    alpha_H=.00005,
    alpha_W=.0005,
    l1_ratio=.5
)
nmf_fn.fit(matrix_tf)
plot_top_words(nmf_fn, feature_names, n_top_features, "Tematu w modeli NMF FN", size, colors)
plot_documents(nmf_fn, matrix_tf, n_components, "Tematu w modeli NF FN", colors)

In [ ]:
nmf_kl = NMF(
    n_components=n_components,
    random_state=1, 
    beta_loss='kullback-leibler',
    solver='mu',
    alpha_H=.00005,
    alpha_W=.0005,
    l1_ratio=.5
)
nmf_kl.fit(matrix_tf)
plot_top_words(nmf_kl, feature_names, n_top_features, "Tematu w modeli NMF KL", size, colors)
plot_documents(nmf_kl, matrix_tf, n_components, "Tematu w modeli NF KL", colors)

# Analiza skupień

In [ ]:
def plot_dendrogram(model, **kwargs):
    # Create linkage matrix and then plot the dendrogram

    # create the counts of samples under each node
    counts = np.zeros(model.children_.shape[0])
    n_samples = len(model.labels_)
    for i, merge in enumerate(model.children_):
        current_count = 0
        for child_idx in merge:
            if child_idx < n_samples:
                current_count += 1  # leaf node
            else:
                current_count += counts[child_idx - n_samples]
        counts[i] = current_count

    linkage_matrix = np.column_stack(
        [model.children_, model.distances_, counts]
    ).astype(float)

    # Plot the corresponding dendrogram
    dendrogram(linkage_matrix, **kwargs)
    
    iris = load_iris()
    
X = iris.dat

# setting distance_threshold=0 ensures we compute the full tree.
model = AgglomerativeClustering(distance_threshold=0, n_clusters=None)

model = model.fit(X)
plt.title("Hierarchical Clustering Dendrogram")
# plot the top three levels of the dendrogram
plot_dendrogram(model, truncate_mode="level", p=3)
plt.xlabel("Number of points in node (or index of point if no parenthesis).")
plt.show()